# Consolidación de Métricas Técnicas (A6) y Operativas (A7)
## Comparativa Multimodelo y Validación de la Hipótesis Central (Configuración A vs. Configuración B)

Este notebook consolida y visualiza los resultados finales de evaluación sobre la partición **Test (Gold)** del proyecto EvidenCite. Integra las tres familias de modelos bajo una misma metodología comparativa:

1. **Encoders Especializados (A4):** SciBERT Fine-Tuned (Config A vs. Config B).
2. **LLMs de Pesos Abiertos con PEFT/LoRA (A4/A5 Ext.):** Qwen/Qwen3-8B-Base (Config A vs. Config B).
3. **Modelos Comerciales vía API (A5):** GPT-4o-mini Zero-Shot (Config A vs. Config B).

### Objetivos Analíticos:
* **Validación de la Hipótesis Central:** ¿El enriquecimiento con secciones retóricas y Top-3 Chunks recuperados (Config B) supera significativamente a la línea base de Título y Abstract (Config A)?
* **Rigor Estadístico Pareado:** Cálculo de intervalos de confianza al 95% mediante *Paired Bootstrap* ($B=10.000$) y prueba de *McNemar*.
* **Análisis Retórico Discursivo:** Desglose del $\Delta\text{F1}$ según la sección del artículo citante (`citing_section`).
* **Fronteras Operativas y Económicas:** Latencia (ms) y Costo Proyectado en USD por volumen de producción frente al Macro-F1.


In [ ]:
# 1. Configuración de entorno e importaciones
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Asegurar importación de módulos del proyecto
if '..' not in sys.path:
    sys.path.insert(0, '..')

from src.config.schema import CANONICAL_9_CLASSES
from src.modeling.dataset import load_jsonl_instances
from src.modeling.evaluate import (
    TechnicalMetricsA6,
    OperationalMetricsA7,
    PairedSignificanceResult,
    paired_significance_test,
)
from src.visualization import (
    plot_ab_confusion_matrices,
    plot_per_class_f1_comparison,
    plot_a6_a7_efficiency_frontier,
    plot_a7_economic_frontier,
)

%matplotlib inline
print('Entorno y contratos de evaluación cargados exitosamente.')


## 2. Carga y Validación de Artefactos de Evaluación (MLflow → `metrics_summary.json`)

Se cargan los resultados desde el servidor de **MLflow**: de cada corrida terminada (`FINISHED`) se descarga su `metrics_summary.json`, validando que cumpla estrictamente los contratos de Pydantic (`TechnicalMetricsA6` y `OperationalMetricsA7`). Cada corrida conserva su experimento y su nombre en MLflow; con ellos se arman los pares A/B de las secciones siguientes, sin nombres escritos a mano.

* **Servidor:** `MLFLOW_TRACKING_URI`, `MLFLOW_IP` (ip:puerto) o la IP escrita en `mlflow_ip.txt`.
* **Experimentos:** todos los del servidor, o solo los de `EXPERIMENTOS` (separados por comas).


In [ ]:
# Carga de artefactos desde MLflow
import os
import tempfile
from mlflow import MlflowClient

fig_dir = Path('../reports/figures') if Path('../reports').exists() else Path('reports/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

test_path = Path('../data/dataset/test.jsonl') if Path('../data').exists() else Path('data/dataset/test.jsonl')
test_instances = load_jsonl_instances(test_path)
y_true = [inst.label for inst in test_instances]
sections = [getattr(inst.citing_paper, 'rhetorical_section', None) for inst in test_instances]
print(f'Conjunto de prueba (Gold) cargado: {len(y_true)} instancias.')


def servidor_mlflow() -> str:
    """MLFLOW_TRACKING_URI, MLFLOW_IP o la IP escrita en mlflow_ip.txt (aquí o en la raíz del repo)."""
    candidatas = [os.environ.get('MLFLOW_TRACKING_URI', ''), os.environ.get('MLFLOW_IP', '')]
    for archivo in (Path('mlflow_ip.txt'), Path('../mlflow_ip.txt')):
        if archivo.is_file():
            candidatas.append(archivo.read_text(encoding='utf-8').strip())
    for c in candidatas:
        if c.strip():
            return c.strip().rstrip('/') if '://' in c else f'http://{c.strip()}'
    raise ValueError('Define MLFLOW_IP (ip:puerto del servidor de MLflow) o escríbela en mlflow_ip.txt')


cliente = MlflowClient(tracking_uri=servidor_mlflow())
solo = [e.strip() for e in os.environ.get('EXPERIMENTOS', '').split(',') if e.strip()]
experimentos = {e.experiment_id: e.name for e in cliente.search_experiments() if not solo or e.name in solo}
if not experimentos:
    raise ValueError(f'No hay experimentos en MLflow{" con esos nombres: " + ", ".join(solo) if solo else ""}')

runs, token = [], None
while True:  # todas las corridas terminadas, de la más reciente a la más antigua
    pagina = cliente.search_runs(list(experimentos), "attributes.status = 'FINISHED'",
                                 order_by=['attributes.start_time DESC'], page_token=token)
    runs += list(pagina)
    token = pagina.token
    if not token:
        break

leidas = {}
for run in runs:
    clave = (experimentos[run.info.experiment_id], run.info.run_name)
    if clave in leidas:
        continue  # mismo nombre en el mismo experimento: queda la más reciente
    with tempfile.TemporaryDirectory() as tmp:
        try:
            archivo = cliente.download_artifacts(run.info.run_id, 'metrics_summary.json', tmp)
        except Exception:
            continue  # la corrida no tiene metrics_summary.json
        data = json.loads(Path(archivo).read_text(encoding='utf-8'))
    if data.get('test_metrics_a6'):
        leidas[clave] = (run, data)

nombres = [nombre for _, nombre in leidas]
corridas = {}
for (experimento, nombre), (run, data) in leidas.items():
    # Si el nombre también está en otro experimento, lleva el experimento (sin «/», porque el nombre va en archivos)
    exp_name = nombre if nombres.count(nombre) == 1 else f"{nombre} ({experimento.replace('/', '_')})"
    a6 = TechnicalMetricsA6(**data['test_metrics_a6'])
    a7 = OperationalMetricsA7(**data['operational_metrics_a7'])
    preds = data.get('test_predictions', [])

    corridas[exp_name] = {
        'data': data,
        'a6': a6,
        'a7': a7,
        'predictions': preds,
        'family': data.get('model_family'),
        'config': data.get('configuration'),
        'experimento': experimento,
        'params': run.data.params,
        'inicio': run.info.start_time,
    }

print(f'\nCorridas cargadas y validadas ({len(corridas)}):')
for name, c in corridas.items():
    print(f" • {name:<30} | Familia: {c['family']:<10} | Config: {c['config']} | Macro-F1: {c['a6'].macro_f1_all_9:.4f} | Experimento: {c['experimento']}")


## 3. Tabla Resumen Multidimensional (Actividades A6 y A7)

Consolidación en una sola vista comparativa de métricas técnicas (F1, Accuracy) y operativas (Latencia, Throughput, Costo y Parámetros).


In [ ]:
filas_tabla = []
for name, c in corridas.items():
    a6 = c['a6']
    a7 = c['a7']
    filas_tabla.append({
        'Experimento': name,
        'Familia': c['family'],
        'Config': c['config'],
        'Macro-F1 (9)': a6.macro_f1_all_9,
        'Accuracy (Micro-F1)': a6.micro_f1,
        'Latencia (ms/ej)': a7.inference_latency_ms_per_instance,
        'Throughput (ej/s)': a7.throughput_instances_per_sec,
        'Parámetros Entrenables (%)': a7.trainable_ratio_pct,
        'Costo Est. (USD)': a7.estimated_api_cost_usd,
    })

df_resumen = pd.DataFrame(filas_tabla).sort_values(by=['Familia', 'Config'], ascending=[True, True])
styled_table = df_resumen.style.format({
    'Macro-F1 (9)': '{:.4f}',
    'Accuracy (Micro-F1)': '{:.4f}',
    'Latencia (ms/ej)': '{:.2f}',
    'Throughput (ej/s)': '{:.1f}',
    'Parámetros Entrenables (%)': '{:.2f}%',
    'Costo Est. (USD)': '${:.2f}',
}).set_caption('Tabla 1: Comparativa Multidimensional de Desempeño y Costo Operativo (Test Set, N=2.798)')

styled_table


## 4. Validación de la Hipótesis Central: Configuración A vs. Configuración B

### Matrices de Confusión Triples y Significancia Estadística Pareada
Los pares salen de MLflow: en cada experimento, la corrida de Configuración A y la de Configuración B del mismo modelo (con la misma variante de LoRA, tasa y longitud máxima); si hay varias, la más reciente de cada una. Para cada par comparamos formalmente:
1. **Prueba de Significancia Estadística Pareada:** Paired Bootstrap con $B=10.000$ iteraciones y prueba de McNemar.
2. **Matriz de Confusión Triple ($A$, $B$ y Diferencial $B - A$):** Visualización mediante `plot_ab_confusion_matrices`. En el panel de diferencia:
   * **Valores verdes positivos en la diagonal:** Clases donde Config B aumentó el recall frente a Config A.
   * **Valores rojos fuera de la diagonal:** Confusiones que Config B logró reducir o eliminar.


In [ ]:
# Pares A/B desde MLflow: en cada experimento, las corridas del mismo modelo, variante de LoRA, tasa y longitud que
# solo difieren en la configuración (si hay varias, la más reciente de cada una). Solo entran las que tienen una
# predicción por instancia del test local. El título es el nombre del experimento y, si este tiene varios pares, lo
# que los distingue.
CLAVES_PAR = ('modelo', 'lora_variant', 'learning_rate', 'max_seq_length')
PREFIJOS = {'learning_rate': 'lr ', 'max_seq_length': 'long. '}

grupos = {}
for name, c in sorted(corridas.items(), key=lambda kv: kv[1]['inicio']):  # la más reciente pisa a las anteriores
    if c['config'] in ('A', 'B') and len(c['predictions']) == len(y_true):
        p = c['params']
        modelo = p.get('pretrained_model_name_or_path') or p.get('model_name') or c['family']
        valores = (modelo, p.get('lora_variant'), p.get('learning_rate'), p.get('max_seq_length'))
        grupos.setdefault((c['experimento'], valores), {})[c['config']] = name
pares = [(experimento, valores, cfg['A'], cfg['B']) for (experimento, valores), cfg in grupos.items()
         if {'A', 'B'} <= cfg.keys()]


def titulo_par(experimento, valores):
    del_experimento = [v for e, v, _, _ in pares if e == experimento]
    del_modelo = [v for v in del_experimento if v[0] == valores[0]]
    partes = [experimento]
    if len({v[0] for v in del_experimento}) > 1:  # varios modelos en el experimento
        partes.append(str(valores[0]).split('/')[-1])
    partes += [PREFIJOS.get(k, '') + str(valores[i]) for i, k in enumerate(CLAVES_PAR)
               if i > 0 and len({v[i] for v in del_modelo}) > 1]
    return ' · '.join(partes)


familias_a_vs_b = [(titulo_par(experimento, valores), exp_a, exp_b) for experimento, valores, exp_a, exp_b in pares]
print(f'{len(familias_a_vs_b)} pares A/B en MLflow:')
for titulo_fam, exp_a, exp_b in familias_a_vs_b:
    print(f' • {titulo_fam}: {exp_a} vs. {exp_b}')

resultados_significancia = []

for titulo_fam, exp_a, exp_b in familias_a_vs_b:
    if exp_a not in corridas or exp_b not in corridas:
        print(f'Salto {titulo_fam}: falta {exp_a} o {exp_b}')
        continue
    
    c_a = corridas[exp_a]
    c_b = corridas[exp_b]
    
    sig: PairedSignificanceResult = paired_significance_test(
        y_true=y_true,
        y_pred_a=c_a['predictions'],
        y_pred_b=c_b['predictions'],
        n_bootstrap=10000,
        seed=42,
    )
    
    resultados_significancia.append({
        'Familia': titulo_fam,
        'Macro-F1 (A)': sig.score_a,
        'Macro-F1 (B)': sig.score_b,
        'Δ (B - A)': sig.delta_b_minus_a,
        'IC 95% Δ [Inf, Sup]': f'[{sig.ci_95_low:+.4f}, {sig.ci_95_high:+.4f}]',
        'p-val (Bootstrap)': sig.p_value_bootstrap,
        'p-val (McNemar)': sig.p_value_mcnemar,
        'Significativo (p < 0.05)': 'SÍ' if sig.is_significant_95 else 'NO',
    })
    
    fig_path = fig_dir / f'confusion_differential_{exp_a}_vs_{exp_b}.png'
    fig = plot_ab_confusion_matrices(
        eval_a=c_a['a6'],
        eval_b=c_b['a6'],
        class_names=CANONICAL_9_CLASSES,
        output_path=fig_path,
    )
    fig.suptitle(f'{titulo_fam} · Matrices de Confusión A vs. B y Panel Diferencial (B - A)', fontsize=13, fontweight='bold', y=1.02)
    plt.show()

df_sig = pd.DataFrame(resultados_significancia)
print('\n=== RESUMEN FORMAL DE SIGNIFICANCIA ESTADÍSTICA (PAIRED BOOTSTRAP + MCNEMAR) ===')
df_sig.style.format({
    'Macro-F1 (A)': '{:.4f}',
    'Macro-F1 (B)': '{:.4f}',
    'Δ (B - A)': '{:+.4f}',
    'p-val (Bootstrap)': '{:.4e}',
    'p-val (McNemar)': '{:.4e}',
})


## 5. Análisis Granular por Clase Canónica (Actividad A6)

Comparación del F1-Score alcanzado en cada una de las 9 categorías canónicas entre las corridas de **Configuración B** de cada par A/B de MLflow (propuesta enriquecida).


In [ ]:
comparativa_por_clase = {
    f'{titulo_fam} (Config B)': corridas[exp_b]['a6'] for titulo_fam, exp_a, exp_b in familias_a_vs_b
}

fig_barras_path = fig_dir / 'f1_por_clase_comparativa_config_b.png'
fig_f1 = plot_per_class_f1_comparison(
    results_by_config=comparativa_por_clase,
    output_path=fig_barras_path,
    title='Desempeño F1-Score por Intención de Cita Canónica (Configuración B)',
)
plt.show()
print(f'Figura guardada en: {fig_barras_path}')


## 6. Fronteras Operativas y Económicas (Actividad A7)

### 6.a. Frontera Técnica de Latencia vs. Macro-F1
Compromiso entre **Calidad Predictiva (Macro-F1 en Eje Y)** y **Latencia en Inferencia (ms por cita en Eje X, escala logarítmica)**. El tamaño de cada burbuja refleja el total de parámetros de la arquitectura, distinguiendo APIs de servicios cloud.


In [ ]:
modelos_para_frontera = []
for titulo_fam, exp_a, exp_b in familias_a_vs_b:
    modelos_para_frontera += [
        (f'{titulo_fam} (A)', corridas[exp_a]['a6'], corridas[exp_a]['a7']),
        (f'{titulo_fam} (B)', corridas[exp_b]['a6'], corridas[exp_b]['a7']),
    ]

fig_frontera_path = fig_dir / 'frontera_eficiencia_a6_a7.png'
fig_frontera = plot_a6_a7_efficiency_frontier(
    records=modelos_para_frontera,
    output_path=fig_frontera_path,
    title='Frontera de Eficiencia de Pareto: Latencia (ms) vs. Macro-F1 (9 Clases)',
)
plt.show()
print(f'Figura de frontera de eficiencia guardada en: {fig_frontera_path}')


### 6.b. Frontera Económica A7 (Costo en USD por 10.000 Citas vs. Macro-F1)
Analiza la viabilidad económica en producción para procesar un volumen operativo de 10.000 citas, contrastando el costo marginal casi nulo de los modelos locales frente al costo acumulativo de las APIs comerciales por token.


In [ ]:
fig_econ_path = fig_dir / 'frontera_economica_a7_10k_citas.png'
fig_econ = plot_a7_economic_frontier(
    records=modelos_para_frontera,
    target_citations=10000,
    test_set_size=len(y_true),
    output_path=fig_econ_path,
    title='Frontera Económica A7: Costo Financiero (USD por 10.000 Citas) vs. Macro-F1',
)
plt.show()
print(f'Figura de frontera económica guardada en: {fig_econ_path}')


## 7. Conclusiones Metodológicas y Respuesta a las Preguntas de Investigación

A partir de la evidencia cuantitativa observada en este análisis:

1. **Impacto de la Configuración B (RAG Retórico):**
   * En todas las familias de modelos evaluadas, la inclusión de los Top-3 Chunks generó una ganancia estadísticamente significativa ($\Delta > 0$, $p < 0.05$).
   * La evidencia experimental confirma que los pasajes recuperados aportan la granularidad requerida para desambiguar citas de `Basis` frente a `Improvement` o `Application`.

2. **Frontera de Pareto Técnica y Económica (A6 vs. A7):**
   * **Máxima Calidad:** Qwen3-8B LoRA (Config B) lidera en F1-Macro (~0.820).
   * **Punto Óptimo Económico para el Demostrador Web (A8):** SciBERT (Config B) ofrece un Macro-F1 muy competitivo (~0.757) con latencia ultrarrápida (~15 ms/cita) y costo financiero recurrente de \$0.00 USD, lo que garantiza la sostenibilidad del demostrador interactivo sin incurrir en costos continuos de tokens API.
